# Dibujo de soportes por clics: revisión de la interfaz

Notebook nuevo (no modifica el ejemplo `agente_estructural.ipynb`). Reúne lo que llevamos avanzado:

- **Dibujo directo por clics** (estilo RAM/SAP): planta X-Y y alzado X-Z, nodos por clic, barras entre nodos, apoyos y rejilla con ajuste.
- **Base de perfiles Gerdau México rev04** (no ICHA). Al seleccionar una barra se ve su sección en el recuadro inferior derecho.
- **Orientación del eje fuerte (peralte)** con la convención del motor: vigas con peralte vertical por defecto, columnas con peralte en Y global. Se muestra la flecha del eje fuerte sobre cada barra.
- Salida `dib.a_modelo()` con el mismo formato que `leer_dxf()`, para conectar luego con el motor de análisis.

**Antes de ejecutar:** subir como dataset de Kaggle el archivo `perfiles_gerdau_mexico_rev04.json`.

**Cómo se usa:** ejecutar las celdas en orden. Los botones de la izquierda cambian el modo: Seleccionar, Nodo, Barra, Poli, Apoyo, Eje, Borrar. Clic derecho cancela. Planta/Alzado cambia la vista. Nivel Z, Cota Y, Perfil y Elemento se escriben en sus campos.

**Limitaciones de esta versión:** la selección de elementos es en vista 2D (planta y alzado); la selección en 3D no está implementada. Solo se modela apoyo empotrado. El cálculo no está en este notebook.


In [ ]:
%pip install -q ipympl
%matplotlib widget

In [ ]:
%%writefile dibujo_directo.py
"""
MODULO DE DIBUJO DIRECTO (reemplaza la lectura de DXF).

Permite construir el modelo estructural dentro del notebook: nodos (puntos x, y, z en m),
barras (polilineas entre nodos), apoyos y ejes fuertes. Produce EXACTAMENTE el mismo
diccionario que entrega lector_validador.leer_dxf(), para que el motor de analisis
(motor_analisis.construir_y_resolver) y el validador funcionen sin cambios.

Formato de salida (igual al DXF):
  {'NODOS': [(x,y,z) en m], 'BARRAS': [{'n1','n2','elemento','perfil','capa','eje_fuerte'}],
   'NODOS_APOYO': [indices], 'ruta': str}

Unidades: metros (la conversion a cm la hace el motor).
"""
import json
import numpy as np

TOL_NODO = 1e-3                 # m: nodos a menos de esta distancia son el mismo nodo
ELEMENTOS = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']
TIPOS_APOYO = ['empotrado']     # el motor actual solo modela empotrado (6 GDL); rotulado/rodillo pendiente


class DibujoModelo:
    def __init__(self, tol=TOL_NODO, ruta='dibujo_directo'):
        self.nodos = []         # lista de tuplas (x, y, z)
        self.barras = []        # lista de dicts con el mismo formato que el DXF
        self.apoyos = {}        # {indice_nodo: tipo}
        self.tol = tol
        self.ruta = ruta

    # ---------- nodos ----------
    def nodo(self, x, y, z):
        """Agrega un nodo (o devuelve el existente a menos de la tolerancia). Devuelve su indice."""
        p = (round(float(x), 4), round(float(y), 4), round(float(z), 4))
        for i, q in enumerate(self.nodos):
            if all(abs(p[k] - q[k]) < self.tol for k in range(3)):
                return i
        self.nodos.append(p)
        return len(self.nodos) - 1

    # ---------- barras ----------
    def _existe(self, n1, n2):
        return any({b['n1'], b['n2']} == {n1, n2} for b in self.barras)

    def barra(self, n1, n2, elemento, perfil):
        """Agrega una barra entre dos nodos (indices). Si ya existe, no la duplica. Devuelve su indice o None."""
        if n1 == n2:
            raise ValueError(f'barra degenerada en el nodo {n1}')
        if self._existe(n1, n2):
            return None
        elemento = str(elemento).upper()
        if elemento not in ELEMENTOS:
            raise ValueError(f'elemento {elemento!r} no reconocido; usar {ELEMENTOS}')
        self.barras.append({'n1': n1, 'n2': n2, 'elemento': elemento, 'perfil': perfil,
                            'capa': f'{elemento}-{perfil}', 'eje_fuerte': None})
        return len(self.barras) - 1

    def polilinea(self, puntos, elemento, perfil):
        """Une una secuencia de puntos con barras consecutivas (estilo 'dibujar con lineas').
        puntos: lista de indices de nodo o de coordenadas (x, y, z). Devuelve indices de barras creadas."""
        ids = [p if isinstance(p, (int, np.integer)) else self.nodo(*p) for p in puntos]
        creadas = []
        for a, b in zip(ids[:-1], ids[1:]):
            r = self.barra(a, b, elemento, perfil)
            if r is not None:
                creadas.append(r)
        return creadas

    def eje_fuerte(self, indice_barra, direccion):
        """Fija la direccion del eje fuerte de una barra (vector en coordenadas globales)."""
        d = np.array(direccion, float)
        n = np.linalg.norm(d)
        if n < 1e-9:
            raise ValueError('direccion nula')
        self.barras[indice_barra]['eje_fuerte'] = tuple(float(v) for v in d / n)

    # ---------- apoyos ----------
    def apoyo(self, nodo_idx, tipo='empotrado'):
        if tipo not in TIPOS_APOYO:
            raise ValueError(f'tipo de apoyo {tipo!r} no soportado por el motor (usar {TIPOS_APOYO})')
        self.apoyos[int(nodo_idx)] = tipo

    # ---------- generadores ----------
    def marco_grilla(self, xs, ys, zs, perfil_columna, perfil_viga_x, perfil_viga_y, eje_fuerte_columna=None):
        """Marco ortogonal a partir de ejes: nodos en todas las intersecciones (x, y, z).
        Columnas en cada (x, y) entre niveles consecutivos; vigas X y vigas Y en cada nivel superior.
        eje_fuerte_columna: direccion del eje fuerte para las columnas (opcional, ej. (1,0,0))."""
        xs, ys, zs = sorted(xs), sorted(ys), sorted(zs)
        idx = {}
        for x in xs:
            for y in ys:
                for z in zs:
                    idx[(x, y, z)] = self.nodo(x, y, z)
        for x in xs:
            for y in ys:
                for za, zb in zip(zs[:-1], zs[1:]):
                    i = self.barra(idx[(x, y, za)], idx[(x, y, zb)], 'COLUMNA', perfil_columna)
                    if i is not None and eje_fuerte_columna is not None:
                        self.eje_fuerte(i, eje_fuerte_columna)
        for y in ys:
            for z in zs[1:]:
                for xa, xb in zip(xs[:-1], xs[1:]):
                    self.barra(idx[(xa, y, z)], idx[(xb, y, z)], 'VIGA', perfil_viga_x)
        for x in xs:
            for z in zs[1:]:
                for ya, yb in zip(ys[:-1], ys[1:]):
                    self.barra(idx[(x, ya, z)], idx[(x, yb, z)], 'VIGA', perfil_viga_y)
        return idx

    # ---------- salida y persistencia ----------
    def a_modelo(self):
        """Diccionario con el formato de leer_dxf(): listo para motor_analisis y validar_modelo."""
        return {'NODOS': list(self.nodos), 'BARRAS': [dict(b) for b in self.barras],
                'NODOS_APOYO': sorted(self.apoyos.keys()), 'ruta': self.ruta}

    def guardar_json(self, ruta):
        datos = {'nodos': self.nodos, 'barras': self.barras,
                 'apoyos': {str(k): v for k, v in self.apoyos.items()}}
        with open(ruta, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=1)

    @classmethod
    def cargar_json(cls, ruta):
        with open(ruta, encoding='utf-8') as f:
            datos = json.load(f)
        m = cls()
        m.nodos = [tuple(p) for p in datos['nodos']]
        m.barras = datos['barras']
        m.apoyos = {int(k): v for k, v in datos['apoyos'].items()}
        return m


def comparar_modelos(m1, m2, tol=1e-3):
    """Compara dos modelos (formato leer_dxf) sin depender del orden de nodos ni barras.
    Devuelve (ok, lista_de_diferencias)."""
    def claves_nodos(m):
        return {tuple(round(c, 3) for c in p) for p in m['NODOS']}

    def claves_barras(m):
        out = []
        for b in m['BARRAS']:
            p1 = tuple(round(c, 3) for c in m['NODOS'][b['n1']])
            p2 = tuple(round(c, 3) for c in m['NODOS'][b['n2']])
            ext = tuple(sorted([p1, p2]))
            ef = None if b['eje_fuerte'] is None else tuple(round(c, 3) for c in b['eje_fuerte'])
            out.append((ext, b['elemento'], b['perfil'], ef))
        return sorted(out, key=str)

    def claves_apoyos(m):
        return sorted(tuple(round(c, 3) for c in m['NODOS'][i]) for i in m['NODOS_APOYO'])

    dif = []
    if claves_nodos(m1) != claves_nodos(m2):
        dif.append(f"nodos distintos: {len(claves_nodos(m1))} vs {len(claves_nodos(m2))}")
    b1, b2 = claves_barras(m1), claves_barras(m2)
    if b1 != b2:
        s1, s2 = set(map(str, b1)), set(map(str, b2))
        dif.append(f"barras distintas: {len(b1)} vs {len(b2)} | solo en 1: {len(s1 - s2)} | solo en 2: {len(s2 - s1)}")
    if claves_apoyos(m1) != claves_apoyos(m2):
        dif.append("apoyos distintos")
    return (len(dif) == 0), dif


In [ ]:
%%writefile perfiles_gerdau.py
"""Adaptador de la base Gerdau Mexico rev04 (dimensiones en cm).
H = d (peralte), B = bf (patin), e = tw (alma), t = tf (patin). Valores nulos quedan None (dato pendiente)."""
import glob
import json

_CANDIDATOS = (glob.glob('/kaggle/input/**/*perfiles_gerdau*.json', recursive=True)
               + glob.glob('/root/.claude/uploads/**/*perfiles_gerdau*.json', recursive=True)
               + glob.glob('*perfiles_gerdau*.json'))
if not _CANDIDATOS:
    raise FileNotFoundError('Suba el JSON perfiles_gerdau_mexico_rev04 como dataset de Kaggle (Add data).')
RUTA = _CANDIDATOS[0]
_DATA = json.load(open(RUTA, encoding='utf-8'))['perfiles']
PERFILES = {p['perfil_comercial']: p for p in _DATA}


def obtener_perfil(nombre):
    p = PERFILES.get(nombre)
    if p is None:
        return None
    return {'familia': p['familia'], 'H': p['d_ala_diametro_cm'], 'B': p['bf_b_cm'],
            'e': p['tw_cm'], 't': p['tf_t_cm'], 'A': p['area_cm2'], 'Ix': p['ix_cm4'], 'Iy': p['iy_cm4'],
            'Sx': p['sx_cm3'], 'Sy': p['sy_cm3'], 'rx': p['rx_cm'], 'ry': p['ry_cm'], 'J': p['j_cm4'],
            'Cw': p['cw_cm6'], 'estado': p['estado']}


In [ ]:
%%writefile dibujo_clics.py
"""
DIBUJO POR CLICS (estilo software de calculo estructural).
Motor de clics independiente de la interfaz: recibe coordenadas de un evento de matplotlib
(event.xdata, event.ydata, event.button) y modifica un dibujo_directo.DibujoModelo.

Vistas:
  - 'planta'   : plano X-Y a una cota Z (nivel actual)
  - 'alzado_y' : plano X-Z a una posicion Y fija
Modos:
  - 'nodo'   : cada clic crea un nodo
  - 'barra'  : dos clics = una barra (el 2do nodo queda como 'pendiente' hasta clic derecho)
  - 'poli'   : clics consecutivos encadenan barras; clic derecho termina
  - 'apoyo'  : clic sobre un nodo lo empotra
  - 'eje'    : clic 1 = seleccionar barra; clic 2 = direccion del eje fuerte
  - 'borrar' : clic sobre una barra la elimina
Ajuste (snap): cada clic se lleva a la interseccion de la rejilla mas cercana si esta dentro de
la tolerancia; si hay un nodo de la vista cerca, se usa ese nodo.
"""
import math
from dibujo_directo import DibujoModelo, ELEMENTOS

MODOS = ['seleccionar', 'nodo', 'barra', 'poli', 'apoyo', 'eje', 'borrar']


class MotorClics:
    def __init__(self, dib, xs, ys, zs, vista='planta', nivel=0.0, cota_y=0.0,
                 elemento='VIGA', perfil='IN30x36.7', modo='barra', tol_frac=0.25):
        self.dib = dib
        self.xs = sorted(xs); self.ys = sorted(ys); self.zs = sorted(zs)
        self.vista = vista; self.nivel = nivel; self.cota_y = cota_y
        self.elemento = elemento; self.perfil = perfil; self.modo = modo
        self.tol_frac = tol_frac
        self.pendiente = None          # indice de nodo de inicio (barra/poli)
        self.seleccion = None          # indice de barra (modo 'eje')
        self.log = []

    def _paso(self, lista):
        if len(lista) < 2:
            return 1.0
        return min(b - a for a, b in zip(lista[:-1], lista[1:]))

    def _ajustar(self, valor, rejilla):
        if not rejilla:
            return round(valor, 3)
        mas_cerca = min(rejilla, key=lambda r: abs(r - valor))
        if abs(mas_cerca - valor) <= self.tol_frac * self._paso(rejilla):
            return mas_cerca
        return round(valor, 3)

    def coordenada(self, xd, yd):
        """Convierte un clic (en unidades de la vista) a (x, y, z) ajustado a la rejilla."""
        if self.vista == 'planta':
            return (self._ajustar(xd, self.xs), self._ajustar(yd, self.ys), self.nivel)
        return (self._ajustar(xd, self.xs), self.cota_y, self._ajustar(yd, self.zs))

    def _en_vista(self, p):
        return abs(p[2] - self.nivel) < 1e-9 if self.vista == 'planta' else abs(p[1] - self.cota_y) < 1e-9

    def _nodo_en_vista(self, xd, yd):
        """Nodo visible cercano al clic (dentro de la tolerancia de la rejilla), o None."""
        mejor, dmin = None, None
        tol_h = self.tol_frac * self._paso(self.xs)
        tol_v = self.tol_frac * self._paso(self.ys if self.vista == 'planta' else self.zs)
        for i, q in enumerate(self.dib.nodos):
            if not self._en_vista(q):
                continue
            hx, vy = (q[0], q[1]) if self.vista == 'planta' else (q[0], q[2])
            if abs(hx - xd) <= tol_h and abs(vy - yd) <= tol_v:
                d = math.hypot(hx - xd, vy - yd)
                if dmin is None or d < dmin:
                    mejor, dmin = i, d
        return mejor

    def _nodo_de_clic(self, xd, yd):
        existente = self._nodo_en_vista(xd, yd)
        if existente is not None:
            return existente
        return self.dib.nodo(*self.coordenada(xd, yd))

    def _barra_cercana(self, xd, yd):
        """Indice de la barra visible mas cercana al clic (dentro de la tolerancia), o None."""
        mejor, dmin = None, None
        for i, b in enumerate(self.dib.barras):
            p1, p2 = self.dib.nodos[b['n1']], self.dib.nodos[b['n2']]
            if not (self._en_vista(p1) and self._en_vista(p2)):
                continue
            ax, ay = (p1[0], p1[1]) if self.vista == 'planta' else (p1[0], p1[2])
            bx, by = (p2[0], p2[1]) if self.vista == 'planta' else (p2[0], p2[2])
            dx, dy = bx - ax, by - ay
            L2 = dx * dx + dy * dy
            t = 0.0 if L2 == 0 else max(0.0, min(1.0, ((xd - ax) * dx + (yd - ay) * dy) / L2))
            d = math.hypot(ax + t * dx - xd, ay + t * dy - yd)
            if dmin is None or d < dmin:
                mejor, dmin = i, d
        tol = self.tol_frac * max(self._paso(self.xs), self._paso(self.ys if self.vista == 'planta' else self.zs))
        return mejor if (dmin is not None and dmin <= tol) else None

    def clic(self, xd, yd, boton=1):
        """Procesa un clic. boton 1 = izquierdo (accion); 3 = derecho (cancelar / terminar)."""
        if boton == 3:
            self.pendiente = None; self.seleccion = None
            self.log.append('cancelado / terminado')
            return self
        if self.modo == 'seleccionar':
            self.seleccion = self._barra_cercana(xd, yd)
            self.log.append(f'seleccionada barra {self.seleccion}' if self.seleccion is not None else 'sin barra')
        elif self.modo == 'nodo':
            self.log.append(f'nodo {self._nodo_de_clic(xd, yd)}')
        elif self.modo in ('barra', 'poli'):
            n = self._nodo_de_clic(xd, yd)
            if self.pendiente is None:
                self.pendiente = n
            elif n != self.pendiente:
                self.dib.barra(self.pendiente, n, self.elemento, self.perfil)
                self.log.append(f'barra {self.pendiente}-{n}')
                self.pendiente = n if self.modo == 'poli' else None
        elif self.modo == 'apoyo':
            n = self._nodo_en_vista(xd, yd)
            if n is not None:
                self.dib.apoyo(n)
                self.log.append(f'apoyo en nodo {n}')
        elif self.modo == 'borrar':
            i = self._barra_cercana(xd, yd)
            if i is not None:
                del self.dib.barras[i]
                self.log.append(f'borrada barra {i}')
        elif self.modo == 'eje':
            if self.seleccion is None:
                self.seleccion = self._barra_cercana(xd, yd)
            else:
                b = self.dib.barras[self.seleccion]
                p1, p2 = self.dib.nodos[b['n1']], self.dib.nodos[b['n2']]
                pm = [(p1[k] + p2[k]) / 2 for k in range(3)]
                # la direccion usa el clic SIN ajustar a la rejilla (es un vector, no un nodo)
                if self.vista == 'planta':
                    d = (xd - pm[0], yd - pm[1], 0.0)
                else:
                    d = (xd - pm[0], 0.0, yd - pm[2])
                self.dib.eje_fuerte(self.seleccion, d)
                self.log.append(f'eje fuerte barra {self.seleccion}')
                self.seleccion = None
        return self


In [ ]:
%%writefile orientacion_gerdau.py
"""
ORIENTACION DEL EJE FUERTE (peralte) EN PANTALLA, con la base Gerdau Mexico rev04.

Convencion del motor (copiada de motor_analisis.py del otro chat, sin depender de ese paquete):
  xAxis = (p2 - p1)/L ; yAxis = vecxz x xAxis ; zAxis = xAxis x yAxis.
  El eje fuerte (Iz) es el eje local y => su direccion es el PERALTE d del perfil.

Defaults del motor (sin marca):
  - vigas/diagonales: peralte VERTICAL (eje fuerte resiste gravedad)
  - columnas:         peralte en Y global (vecxz = (1,0,0))
"""
import math
import numpy as np
import perfiles_gerdau as G


def vector_vecxz(p1, p2, elemento):
    """vecxz por defecto segun el tipo de elemento (igual que el motor)."""
    v = np.array(p2, float) - np.array(p1, float)
    L = np.linalg.norm(v)
    x = v / L
    if abs(x[2]) > 0.9:                       # columna (casi vertical)
        return (1.0, 0.0, 0.0)
    vx = np.cross(x, np.array([0.0, 0.0, 1.0]))   # viga/diagonal: yAxis vertical
    n = np.linalg.norm(vx)
    if n < 1e-6:
        return (1.0, 0.0, 0.0)
    vx = vx / n
    return (float(vx[0]), float(vx[1]), float(vx[2]))


def vecxz_desde_direccion(p1, p2, s):
    """vecxz que hace que el eje local y (= peralte) apunte segun s (proyectado perpendicular a la barra)."""
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    s = np.array(s, float)
    u = s - x * np.dot(s, x)
    nu = np.linalg.norm(u)
    if nu < 1e-6:
        return None
    u = u / nu
    v = np.cross(x, u)
    return (float(v[0]), float(v[1]), float(v[2]))


def vecxz_de_barra(nodos, b):
    """vecxz de una barra: usa 'eje_fuerte' si existe, si no el default por tipo de elemento."""
    p1 = nodos[b['n1']]; p2 = nodos[b['n2']]
    s = b.get('eje_fuerte')
    if s:
        v = vecxz_desde_direccion(p1, p2, s)
        if v is not None:
            return v
    return vector_vecxz(p1, p2, b.get('elemento'))


def ejes_locales(dib, i):
    """Ejes locales (x, y, z) de la barra i."""
    b = dib.barras[i]
    p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
    vecxz = np.array(vecxz_de_barra(dib.nodos, b), float)
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    y = np.cross(vecxz, x); y = y / np.linalg.norm(y)
    z = np.cross(x, y)
    return x, y, z


def dimensiones(perfil):
    """(d, bf) en metros desde Gerdau (H = d, B = bf en cm). None si el perfil no existe o no tiene dato."""
    p = G.obtener_perfil(perfil)
    if p is None or p['H'] is None or p['B'] is None:
        return None
    return float(p['H']) / 100.0, float(p['B']) / 100.0


def contorno(dib, i, escala=1.0):
    """4 vertices globales del rectangulo del perfil en el punto medio de la barra i.
    Alto = d a lo largo del eje local y (eje fuerte); ancho = bf a lo largo de z."""
    b = dib.barras[i]
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    pm = (p1 + p2) / 2
    dims = dimensiones(b['perfil'])
    if dims is None:
        return None
    d, bf = dims[0] * escala, dims[1] * escala
    x, y, z = ejes_locales(dib, i)
    c = [pm + sy * (d / 2) * y + sz * (bf / 2) * z for sy, sz in [(1, 1), (-1, 1), (-1, -1), (1, -1)]]
    return [tuple(float(v) for v in q) for q in c]


PRESETS = {'peralte vertical (Z)': (0.0, 0.0, 1.0),
           'peralte en X': (1.0, 0.0, 0.0),
           'peralte en Y': (0.0, 1.0, 0.0)}


def orientar(dib, i, preset):
    """Fija el peralte (eje fuerte) de la barra i con un preset o un vector. Rechaza direccion paralela a la barra."""
    b = dib.barras[i]
    s = np.array(PRESETS[preset] if isinstance(preset, str) else preset, float)
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    if abs(np.dot(s, x)) > 0.999:
        raise ValueError(f'el peralte no puede ser paralelo a la barra {i}; elija otra direccion')
    dib.eje_fuerte(i, tuple(s))
    return b['eje_fuerte']


def ortogonalidad(dib, i):
    """Angulo (grados) entre el eje fuerte asignado y la barra; 90 = perpendicular (correcto)."""
    b = dib.barras[i]
    if b['eje_fuerte'] is None:
        return None
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    s = np.array(b['eje_fuerte'], float)
    return math.degrees(math.acos(max(-1.0, min(1.0, abs(np.dot(s, x))))))


In [ ]:
%%writefile interfaz_clics.py
"""
INTERFAZ DE DIBUJO POR CLICS (matplotlib: funciona en Kaggle con %matplotlib widget / ipympl,
y en cualquier entorno para revisar la pantalla).

Layout:
  izquierda : modos (Nodo, Barra, Polilinea, Apoyo, Eje fuerte, Borrar), vista (Planta/Alzado),
              nivel Z, cota Y, perfil, elemento
  centro    : vista de dibujo con rejilla, nodos, barras por elemento, apoyos y ejes fuertes
  derecha   : resumen (nodos, barras, apoyos), modo activo, pendiente y ultimas acciones
"""
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, TextBox, RadioButtons
from dibujo_clics import MotorClics, MODOS
from dibujo_directo import ELEMENTOS
import numpy as np
import perfiles_gerdau as G


def _seccion_poligono(familia, H, B, tw, tf):
    """Contorno de la seccion en coordenadas (horizontal = eje debil, vertical = peralte)."""
    h, b = H / 2, B / 2
    if familia == 'CE':   # canal: alma a la izquierda, alas hacia +horizontal
        return [(-b, -h), (b, -h), (b, -h + tf), (-b + tw, -h + tf), (-b + tw, h - tf), (b, h - tf), (b, h), (-b, h)]
    if familia == 'IR':   # I laminada
        w = tw / 2
        return [(-b, -h), (b, -h), (b, -h + tf), (w, -h + tf), (w, h - tf), (b, h - tf), (b, h), (-b, h),
                (-b, h - tf), (-w, h - tf), (-w, -h + tf), (-b, -h + tf)]
    return None

COLOR = {'COLUMNA': '#1f77b4', 'VIGA': '#2ca02c', 'ARRIOSTRE': '#d62728', 'DIAGONAL': '#ff7f0e'}


class InterfazClics:
    def __init__(self, dib, xs, ys, zs, perfil='IR 12 × 16', elemento='VIGA', nivel=0.0, cota_y=0.0):
        self.dib = dib
        self.motor = MotorClics(dib, xs, ys, zs, vista='planta', nivel=nivel, cota_y=cota_y,
                                elemento=elemento, perfil=perfil, modo='barra')
        self.xs, self.ys, self.zs = sorted(xs), sorted(ys), sorted(zs)
        self.log = []
        self.fig = plt.figure(figsize=(14, 8))
        self.fig.canvas.manager.set_window_title('Dibujo por clics') if hasattr(self.fig.canvas, 'manager') and self.fig.canvas.manager else None
        self.ax = self.fig.add_axes([0.25, 0.08, 0.53, 0.86])
        self._construir_botones()
        self.info = self.fig.text(0.80, 0.97, '', va='top', family='monospace', fontsize=8)
        self.ax_sec = self.fig.add_axes([0.84, 0.04, 0.12, 0.18])
        self.ax_sec.set_visible(False)
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self._on_click)
        self.redibujar()

    # ---------- botones y campos ----------
    def _construir_botones(self):
        self.btn = {}
        y0 = 0.93
        for i, m in enumerate(MODOS):
            ax_b = self.fig.add_axes([0.03, y0 - 0.045 * i, 0.17, 0.036])
            b = Button(ax_b, m.capitalize())
            b.on_clicked(lambda _e, m=m: self.set_modo(m))
            self.btn[m] = b
        ax_v = self.fig.add_axes([0.03, 0.62, 0.17, 0.036])
        self.btn_vista = Button(ax_v, 'Vista: planta')
        self.btn_vista.on_clicked(lambda _e: self.toggle_vista())
        ax_n = self.fig.add_axes([0.13, 0.54, 0.07, 0.03])
        self.tb_nivel = TextBox(ax_n, 'Nivel Z (m) ', initial=str(self.motor.nivel))
        self.tb_nivel.on_submit(self._set_nivel)
        ax_c = self.fig.add_axes([0.13, 0.48, 0.07, 0.03])
        self.tb_cota = TextBox(ax_c, 'Cota Y (m) ', initial=str(self.motor.cota_y))
        self.tb_cota.on_submit(self._set_cota)
        ax_p = self.fig.add_axes([0.13, 0.42, 0.07, 0.03])
        self.tb_perfil = TextBox(ax_p, 'Perfil ', initial=self.motor.perfil)
        self.tb_perfil.on_submit(self._set_perfil)
        ax_e = self.fig.add_axes([0.03, 0.22, 0.17, 0.16])
        self.rb_elem = RadioButtons(ax_e, ELEMENTOS, active=ELEMENTOS.index(self.motor.elemento))
        self.rb_elem.on_clicked(self._set_elemento)
        ax_r = self.fig.add_axes([0.03, 0.10, 0.17, 0.04])
        self.btn_reset = Button(ax_r, 'Reiniciar modelo')
        self.btn_reset.on_clicked(lambda _e: self._reiniciar())

    # ---------- acciones ----------
    def set_modo(self, m):
        self.motor.modo = m; self.motor.pendiente = None; self.motor.seleccion = None
        self._registrar(f'modo: {m}'); self.redibujar()

    def toggle_vista(self):
        self.motor.vista = 'alzado_y' if self.motor.vista == 'planta' else 'planta'
        self.btn_vista.label.set_text('Vista: ' + ('alzado Y' if self.motor.vista == 'alzado_y' else 'planta'))
        self.motor.pendiente = None
        self._registrar(f'vista: {self.motor.vista}'); self.redibujar()

    def _set_nivel(self, txt):
        try:
            self.motor.nivel = float(txt.replace(',', '.'))
        except ValueError:
            return
        self._registrar(f'nivel Z = {self.motor.nivel}'); self.redibujar()

    def _set_cota(self, txt):
        try:
            self.motor.cota_y = float(txt.replace(',', '.'))
        except ValueError:
            return
        self._registrar(f'cota Y = {self.motor.cota_y}'); self.redibujar()

    def _set_perfil(self, txt):
        self.motor.perfil = txt.strip(); self._registrar(f'perfil = {self.motor.perfil}'); self.redibujar()

    def _set_elemento(self, label):
        self.motor.elemento = label; self._registrar(f'elemento = {label}'); self.redibujar()

    def _reiniciar(self):
        self.dib.nodos.clear(); self.dib.barras.clear(); self.dib.apoyos.clear()
        self.motor.pendiente = None; self.motor.seleccion = None
        self._registrar('modelo reiniciado'); self.redibujar()

    def _registrar(self, texto):
        self.log.append(texto); self.log = self.log[-4:]

    # ---------- clics ----------
    def _on_click(self, event):
        if event.inaxes is not self.ax or event.xdata is None:
            return
        n_antes = len(self.motor.log)
        self.motor.clic(event.xdata, event.ydata, event.button)
        for linea in self.motor.log[n_antes:]:
            self._registrar(linea)
        self.redibujar()

    def simular_clic(self, xd, yd, boton=1):
        """Para pruebas: ejecuta el mismo camino que un clic real."""
        class _E: pass
        e = _E(); e.inaxes = self.ax; e.xdata = xd; e.ydata = yd; e.button = boton
        self._on_click(e)

    # ---------- dibujo ----------
    def _ejes_vista(self):
        if self.motor.vista == 'planta':
            return self.xs, self.ys, self.motor.nivel, 'X (m)', 'Y (m)'
        return self.xs, self.zs, self.motor.cota_y, 'X (m)', 'Z (m)'

    def redibujar(self):
        ax = self.ax; ax.cla()
        xs, rej_v, plano, lx, ly = self._ejes_vista()
        dib = self.dib; N = dib.nodos
        def en_plano(p):
            return abs(p[2] - plano) < 1e-9 if self.motor.vista == 'planta' else abs(p[1] - plano) < 1e-9
        def coord(p):
            return (p[0], p[1]) if self.motor.vista == 'planta' else (p[0], p[2])
        # rejilla
        for x in xs: ax.axvline(x, color='#bbb', lw=0.8, zorder=0)
        for v in rej_v: ax.axhline(v, color='#bbb', lw=0.8, zorder=0)
        ax.axhline(0, color='#eee', lw=0.5, zorder=0)
        # barras del plano
        for i, b in enumerate(dib.barras):
            p1, p2 = N[b['n1']], N[b['n2']]
            if not (en_plano(p1) and en_plano(p2)):
                continue
            (x1, y1), (x2, y2) = coord(p1), coord(p2)
            ax.plot([x1, x2], [y1, y2], color=COLOR[b['elemento']], lw=3.5, zorder=2, solid_capstyle='round')
            if b['eje_fuerte'] is not None:
                mx, my = (x1 + x2) / 2, (y1 + y2) / 2
                dx, dy = (b['eje_fuerte'][0], b['eje_fuerte'][1]) if self.motor.vista == 'planta' else (b['eje_fuerte'][0], b['eje_fuerte'][2])
                n = (dx * dx + dy * dy) ** 0.5
                if n > 1e-9:
                    s = 0.12 * max(max(xs) - min(xs), 1)
                    ax.annotate('', xy=(mx + s * dx / n, my + s * dy / n), xytext=(mx, my),
                                arrowprops=dict(arrowstyle='-|>', color='magenta', lw=2), zorder=4)
        # nodos del plano
        ids_plano = [i for i, p in enumerate(N) if en_plano(p)]
        for i in ids_plano:
            x, y = coord(N[i])
            ax.plot(x, y, 'o', color='k', ms=4, zorder=3)
            ax.text(x, y, f' {i}', fontsize=7, va='bottom', zorder=3)
        # apoyos
        for i in dib.apoyos:
            if i in ids_plano:
                x, y = coord(N[i]); ax.plot(x, y, 's', color='red', ms=9, mfc='none', mew=2, zorder=5)
        # pendiente
        if self.motor.pendiente is not None and self.motor.pendiente < len(N) and en_plano(N[self.motor.pendiente]):
            x, y = coord(N[self.motor.pendiente])
            ax.plot(x, y, 'o', mfc='none', mec='orange', ms=14, mew=2.5, zorder=6)
        ax.set_xlim(min(xs) - 1.5, max(xs) + 1.5)
        ax.set_ylim(min(rej_v) - 1.2, max(rej_v) + 1.2)
        ax.set_aspect('equal'); ax.set_xlabel(lx); ax.set_ylabel(ly)
        titulo = (f"Planta — nivel Z = {plano:g} m" if self.motor.vista == 'planta' else f"Alzado — cota Y = {plano:g} m")
        ax.set_title(titulo + f"   |   modo: {self.motor.modo}", fontsize=11)
        # barra seleccionada: resaltado y propiedades de la base Gerdau
        self.ax_sec.cla(); self.ax_sec.set_visible(False)
        det = ''
        sel = self.motor.seleccion
        if sel is not None and sel < len(dib.barras):
            b = dib.barras[sel]
            p1, p2 = N[b['n1']], N[b['n2']]
            (x1, y1), (x2, y2) = coord(p1), coord(p2)
            if en_plano(p1) and en_plano(p2):
                ax.plot([x1, x2], [y1, y2], color='k', lw=7, alpha=0.25, zorder=1)
            per = G.obtener_perfil(b['perfil'])
            if per is None:
                det = f"Barra {sel}: perfil '{b['perfil']}' no esta en la base Gerdau"
            else:
                det = (f"Barra {sel}  ({b['elemento']})\n{b['perfil']}\nfamilia: {per['familia']}\n"
                       f"d={per['H']:g} bf={per['B']:g} tw={per['e']:g} tf={per['t']:g} cm\n"
                       f"A={per['A']:g} cm2  Ix={per['Ix']:g}  Iy={per['Iy']:g} cm4\n"
                       f"eje fuerte: {'asignado' if b['eje_fuerte'] else 'por defecto del motor'}")
                pol = _seccion_poligono(per['familia'], per['H'], per['B'], per['e'], per['t'])
                if pol:
                    self.ax_sec.set_visible(True)
                    self.ax_sec.fill([q[0] for q in pol] + [pol[0][0]], [q[1] for q in pol] + [pol[0][1]],
                                     color='#9ecae1', edgecolor='k')
                    self.ax_sec.annotate('', xy=(0, per['H'] / 2 + 3), xytext=(0, 0),
                                         arrowprops=dict(arrowstyle='-|>', color='magenta', lw=2.5))
                    self.ax_sec.text(0.6, per['H'] / 2 + 1.5, 'eje fuerte', color='magenta', fontsize=7)
                    self.ax_sec.set_aspect('equal'); self.ax_sec.set_xticks([]); self.ax_sec.set_yticks([])
                    self.ax_sec.set_xlim(-per['B'] / 2 - 2, per['B'] / 2 + 3)
                    self.ax_sec.set_ylim(-per['H'] / 2 - 2, per['H'] / 2 + 5)
                    self.ax_sec.set_title('seccion', fontsize=8)
        # panel de informacion
        apoyos = len(dib.apoyos)
        txt = (f"Nodos: {len(N)}\nBarras: {len(dib.barras)}\nApoyos: {apoyos}\n\n"
               f"Modo: {self.motor.modo}\nPendiente: {self.motor.pendiente if self.motor.pendiente is not None else '-'}\n"
               f"Perfil: {self.motor.perfil}\nElemento: {self.motor.elemento}\n\n{det}\n\nUltimas acciones:\n" + '\n'.join('  ' + l for l in self.log))
        self.info.set_text(txt)
        self.fig.canvas.draw_idle()


In [ ]:
# ---- Demo: marco de 3 x 2 ejes, 3 niveles; apoyos en la base; orientacion y pantalla ----
import matplotlib.pyplot as plt
from dibujo_directo import DibujoModelo
from orientacion_gerdau import orientar, ortogonalidad, contorno
from interfaz_clics import InterfazClics

xs, ys, zs = [0, 5, 10], [0, 4], [0, 3.5, 7]
dib = DibujoModelo(ruta='soportes_demo')
dib.marco_grilla(xs, ys, zs, 'IR 10 × 30', 'IR 12 × 16', 'IR 8 × 28')
for i, p in enumerate(dib.nodos):
    if abs(p[2]) < 1e-9:
        dib.apoyo(i)
print('nodos', len(dib.nodos), '| barras', len(dib.barras), '| apoyos', len(dib.apoyos))

# Peralte de una viga: vertical (Z) por defecto del motor; se fija explicitamente como ejemplo
i_viga = next(i for i, b in enumerate(dib.barras) if b['elemento'] == 'VIGA')
orientar(dib, i_viga, 'peralte vertical (Z)')
print('viga', i_viga, '| angulo eje fuerte vs barra =', round(ortogonalidad(dib, i_viga), 1), 'grados')
print('contorno de la seccion (4 vertices):', contorno(dib, i_viga) is not None)

modelo = dib.a_modelo()          # formato leer_dxf(): listo para el motor de analisis
ui = InterfazClics(dib, xs, ys, zs, perfil='IR 12 × 16', elemento='VIGA', nivel=3.5)
plt.show()
